## Load Data

In [ ]:
import pandas as pd

df = pd.read_csv('TMDB_female_representation.csv')
df.shape
df.head()
df.info()
df.describe()
df['representation_tier'].value_counts()

## Dataframe Finalization

In [ ]:
# ─────────────────────────────────────────────────────────────
# DATAFRAME FINALIZATION
# ─────────────────────────────────────────────────────────────

# 1. DROP budget, revenue, runtime (too many zeros)
cols_to_drop = ['budget', 'revenue', 'runtime']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f'Dropped columns: {cols_to_drop}')

# 2. GENRE REMAPPING — top 10 + remap 4 rare genres
TOP_10_GENRES = [
    'Drama', 'Comedy', 'Thriller', 'Action', 'Romance',
    'Horror', 'Crime', 'Adventure', 'Family', 'Science Fiction'
]

GENRE_REMAP = {
    'Fantasy':     'Adventure',
    'Animation':   'Family',
    'Mystery':     'Thriller',
    'Documentary': 'Drama',
}

def assign_primary_genre(genre_str):
    if not isinstance(genre_str, str):
        return None
    genres = [g.strip() for g in genre_str.split('|')]
    for genre in genres:
        if genre in TOP_10_GENRES:
            return genre
    for genre in genres:
        if genre in GENRE_REMAP:
            return GENRE_REMAP[genre]
    return None

df['primary_genre'] = df['genre_names'].apply(assign_primary_genre)

before = len(df)
df = df.dropna(subset=['primary_genre'])
after = len(df)
print(f'Dropped {before - after} rows with no genre match')

# ─────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────────────────────
print(f"\n{'='*40}")
print(f'FINAL DATAFRAME: {len(df)} rows x {len(df.columns)} columns')

print(f'\nPrimary genre distribution:')
counts = df['primary_genre'].value_counts()
for genre, count in counts.items():
    print(f'  {genre:20s}: {count:5d} ({100*count/len(df):.1f}%)')

print(f'\nRepresentation features - missing values:')
rep_features = [
    'lead_gender', 'lead_actor_age', 'pct_female_top5',
    'pct_female_cast', 'director_gender', 'director_age',
    'pct_female_writers', 'pct_female_producers'
]
for f in rep_features:
    if f in df.columns:
        missing = df[f].isna().sum()
        print(f'  {f:30s}: {missing} missing ({100*missing/len(df):.1f}%)')

print(f'\nlead_gender distribution (including unknowns):')
print(df['lead_gender'].value_counts())

## Genre Distribution Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 5))
counts = df['primary_genre'].value_counts()
colors = ['#5b8dd9'] * len(counts)

bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white')

for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count:,}\n({100*count/len(df):.1f}%)',
            ha='center', va='bottom', fontsize=8)

ax.set_title('Distribution of Primary Genres (After Remapping)', fontsize=13, fontweight='bold')
ax.set_xlabel('Genre')
ax.set_ylabel('Number of Films')
ax.set_ylim(0, counts.max() * 1.25)
ax.set_xticklabels(counts.index, rotation=45, ha='right')
sns.despine()
plt.tight_layout()
plt.show()